# Dataset Visualization

This notebook demonstrates how to:

- Loading train and validation datasets
- Printing dataset statistics (problems, states)
- Inspecting sample structure and shapes
- Visualizing scenes and trajectories

## 1. Setup Environment and Imports

In [1]:
%cd /workspace
import sys
import os

# Set PYTHONPATH for robofin
robofin_path = os.path.join(os.getcwd(), 'robofin')
current_pythonpath = os.environ.get('PYTHONPATH', '')
if robofin_path not in current_pythonpath:
    os.environ['PYTHONPATH'] = f"{robofin_path}:{current_pythonpath}" if current_pythonpath else robofin_path

# Also add to sys.path for immediate effect
if robofin_path not in sys.path:
    sys.path.insert(0, robofin_path)

/workspace


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
import torch
import numpy as np
import yaml
from pathlib import Path
from torch.utils.data import DataLoader

# Import dataset and robot modules
from avoid_everything.data_loader import TrajectoryDataset, StateDataset
from avoid_everything.type_defs import DatasetType
from robofin.robots import Robot
from utils import visualization
import viz_client

print("✓ All imports successful!")

/home/fishbotics/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


✓ All imports successful!


## 2. Load Configuration and Robot

In [3]:
# Load configuration from evaluation.yaml
config_path = "/workspace/model_configs/evaluation.yaml"
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  Data directory: {cfg['data_module_parameters']['data_dir']}")
print(f"  Robot URDF: {cfg['shared_parameters']['urdf_path']}")
print(f"  Robot points: {cfg['shared_parameters']['num_robot_points']}")
print(f"  Obstacle points: {cfg['data_module_parameters']['num_obstacle_points']}")
print(f"  Target points: {cfg['data_module_parameters']['num_target_points']}")

Configuration loaded:
  Data directory: /workspace/datasets/ae_aristotle1_5mm_cubbies
  Robot URDF: /workspace/assets/panda/panda_spheres.urdf
  Robot points: 2048
  Obstacle points: 4096
  Target points: 128


In [4]:
# Initialize robot
urdf_path = cfg['shared_parameters']['urdf_path']
robot = Robot(urdf_path, device='cpu')

print(f"Robot name: {robot.name}")
print(f"DOF: {robot.MAIN_DOF}")
print(f"DOF including auxiliary joints: {robot.DOF}")
print(f"Arm links: {robot.arm_link_names}")
print(f"End-effector link: {robot.tcp_link_name}")
# print joints
print(f"Joint names: {robot.main_joint_names}")
print(f"Auxiliary joint names: {robot.auxiliary_joint_names}")
print(f"Joint limits: {robot.joint_limits}")


Robot name: panda
DOF: 7
DOF including auxiliary joints: 8
Arm links: ['panda_link0', 'panda_link1', 'panda_link2', 'panda_link3', 'panda_link4', 'panda_link5', 'panda_link6', 'panda_link7', 'panda_link8', 'panda_hand', 'panda_leftfinger', 'panda_rightfinger', 'right_gripper', 'panda_grasptarget']
End-effector link: right_gripper
Joint names: ['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']
Auxiliary joint names: ['panda_finger_joint1', 'panda_finger_joint2']
Joint limits: [[-2.9671  2.9671]
 [-1.8326  1.8326]
 [-2.9671  2.9671]
 [-3.1416  0.    ]
 [-2.9671  2.9671]
 [-0.0873  3.8223]
 [-2.9671  2.9671]
 [ 0.      0.04  ]
 [ 0.      0.04  ]]


## 3. Load Datasets and Print Statistics

Load StateDataset (usually for Behavior Cloning training):

"This is the dataset used primarily for training. Each element in the dataset
    represents the robot and scene at a particular time $t$. Likewise, the
    supervision is the robot's configuration at q_{t+1}."

In [ ]:
train_dataset = StateDataset.load_from_directory(
    robot=robot,
    directory=cfg['data_module_parameters']['data_dir'],
    dataset_type=DatasetType.TRAIN,
    trajectory_key=cfg['data_module_parameters']['train_trajectory_key'],
    num_robot_points=cfg['shared_parameters']['num_robot_points'],
    num_obstacle_points=cfg['data_module_parameters']['num_obstacle_points'],
    num_target_points=cfg['data_module_parameters']['num_target_points'],
    random_scale=cfg['data_module_parameters']['random_scale'], # add random noise to joints
    action_chunk_length=1
)

print("TRAINING DATASET\n================")
print(f"Number of expert demonstrations: {train_dataset.problem_count:,}")
print(f"Number of states: {train_dataset.state_count:,}")
print(f"Location: {train_dataset._database}")

TRAINING DATASET
Number of expert demonstrations: 1,249,258
Number of states: 37,738,869
Location: /workspace/datasets/ae_aristotle1_5mm_cubbies/train/train.hdf5


Load TrajectoryDataset:

"This dataset is used exclusively for validating. Each element in the dataset
    represents a trajectory start and scene. There is no supervision because
    this is used to produce an entire rollout and check for success. When doing
    validation, we care more about success than we care about matching the
    expert's behavior (which is a key difference from training)."

In [6]:
# Load validation dataset
val_dataset = TrajectoryDataset.load_from_directory(
    robot=robot,
    directory=cfg['data_module_parameters']['data_dir'],
    dataset_type=DatasetType.VAL,
    trajectory_key=cfg['data_module_parameters']['val_trajectory_key'],
    num_robot_points=cfg['shared_parameters']['num_robot_points'],
    num_obstacle_points=cfg['data_module_parameters']['num_obstacle_points'],
    num_target_points=cfg['data_module_parameters']['num_target_points'],
    random_scale=0.0  # No noise for validation
)

print("VALIDATION DATASET\n==================")
print(f"Number of expert demonstrations: {val_dataset.problem_count:,}")
print(f"Number of states: {val_dataset.state_count:,}")
print(f"Trajectory average length: {val_dataset.state_count / val_dataset.problem_count:.2f} states")
print(f"Location: {val_dataset._database}")

VALIDATION DATASET
Number of expert demonstrations: 10,034
Number of states: 303,282
Trajectory average length: 30.23 states
Location: /workspace/datasets/ae_aristotle1_5mm_cubbies/val/val.hdf5


## 4. Inspect a Sample from the Dataset

In [8]:
# Get one sample
sample = val_dataset[0]  

print("Sample keys:")
for key, value in sample.items():
    if isinstance(value, torch.Tensor):
        print(f" - {key:20s}: shape {list(value.shape)}, dtype {value.dtype}")
    else:
        print(f" - {key:20s}: {type(value)}")

Sample keys:
 - target_position     : shape [3], dtype torch.float32
 - target_orientation  : shape [3, 3], dtype torch.float32
 - cuboid_dims         : shape [8, 3], dtype torch.float32
 - cuboid_centers      : shape [8, 3], dtype torch.float32
 - cuboid_quats        : shape [8, 4], dtype torch.float32
 - cylinder_radii      : shape [0, 1], dtype torch.float32
 - cylinder_heights    : shape [0, 1], dtype torch.float32
 - cylinder_centers    : shape [0, 3], dtype torch.float32
 - cylinder_quats      : shape [0, 4], dtype torch.float32
 - point_cloud         : shape [6272, 3], dtype torch.float32
 - point_cloud_labels  : shape [6272, 1], dtype torch.float32
 - configuration       : shape [7], dtype torch.float32
 - expert              : shape [60, 7], dtype torch.float32
 - pidx                : shape [], dtype torch.int64


In [9]:
# Print target information
print("Target End-Effector Pose:")
print(f"  Position: {sample['target_position'].numpy()}")
print(f"  Orientation (rotation matrix):")
print(f"    {sample['target_orientation'].numpy()}")

# Print obstacle information
print(f"\nObstacles:")
print(f"  Cuboids: {sample['cuboid_centers'].shape[0]}")
print(f"    - Centers shape: {sample['cuboid_centers'].shape}")
print(f"    - Dimensions shape: {sample['cuboid_dims'].shape}")
print(f"  Cylinders: {sample['cylinder_centers'].shape[0]}")
print(f"    - Centers shape: {sample['cylinder_centers'].shape}")
print(f"    - Radii shape: {sample['cylinder_radii'].shape}")
print(f"    - Heights shape: {sample['cylinder_heights'].shape}")

# Print robot configuration
print(f"\nRobot Configuration:")
print(f"  Joint angles (normalized): {sample['configuration'].numpy()}")
print(f"  Robot point cloud shape: {sample['point_cloud'].shape}")

Target End-Effector Pose:
  Position: [ 0.7018711  -0.47278112  0.5011363 ]
  Orientation (rotation matrix):
    [[ 5.0033053e-05 -1.5007661e-01  9.8867434e-01]
 [ 5.0383683e-06  9.8867434e-01  1.5007661e-01]
 [-1.0000000e+00 -2.5274853e-06  5.0222538e-05]]

Obstacles:
  Cuboids: 8
    - Centers shape: torch.Size([8, 3])
    - Dimensions shape: torch.Size([8, 3])
  Cylinders: 0
    - Centers shape: torch.Size([0, 3])
    - Radii shape: torch.Size([0, 1])
    - Heights shape: torch.Size([0, 1])

Robot Configuration:
  Joint angles (normalized): [-0.532265    0.71754634  0.58597314 -0.02940089 -0.03159714  0.44408143
 -0.13361663]
  Robot point cloud shape: torch.Size([6272, 3])


## 5. Visualize dataset

This will launch the visualization server and display:
- Robot at start configuration (solid)
- Target end-effector pose (purple ghost)
- Obstacle cuboids and cylinders (reddish)

Connect to viz server and publish robot in neutral configuration:

In [10]:
# Connect to visualization server
viz_client.connect(str(robot.urdf_path))
viz_client.publish_joints(joints=robot.neutral_config_dict)

viz_server not running — starting…
Connected to viz_server


Visualize single sample: robot in start config, obstacles and target end-effector pose.

In [11]:
# Visualize single sample 
# from val_dataset so that no noise is added to the joints and the rendered robot is deterministic 
sample = val_dataset[0] 

visualization.visualize_problem(
    robot=robot,
    sample=sample,
    target_alpha=0.7,
    target_color=[0.8, 0.2, 0.8],  # Purple for target
    obstacle_color=[0.8, 0.5, 0.6]  # Reddish for obstacles
)

print("✓ Visualization published!")
print("  Check RViz or Foxglove to see the visualization.")

✓ Visualization published!
  Check RViz or Foxglove to see the visualization.


Visualize multiple samples using a slider:

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create the widget objects
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(val_dataset) - 1, # Dynamically set max based on your dataset size
    step=1,
    description='Sample Index:',
    continuous_update=False # Highly recommended for ROS/Robot vis
)

output = widgets.Output()

# Callback
def on_value_change(change):
    # 'change' is a dict. The actual number is in change['new']
    index = change['new']
    
    with output:
        clear_output(wait=True)
        print(f"Visualizing Sample: {index}")
        
        # Extract the specific sample from the dataset
        sample_i = val_dataset[index]  
        # Trigger ROS/Robot visualization
        visualization.visualize_problem(robot=robot, sample=sample_i)
        
        config_str = ", ".join([f"{angle:.3f}" for angle in sample_i['configuration'].numpy()])
        print(f"Robot configuration: [{config_str}]")
        print(f"✓ Visualization updated!")

# Link the slider to the function
slider.observe(on_value_change, names='value')
# Display
display(slider, output)
# Trigger the first sample immediately so the screen isn't blank
on_value_change({'new': slider.value})

IntSlider(value=0, continuous_update=False, description='Sample Index:', max=10033)

Output()

Target pose as 4x4 homogeneous transformation matrix:
+ 3x3 rotation matrix in the top-left
+ 3x1 translation vector in the top-right

In [13]:
demo_sample = val_dataset[0]

# homogeneous transformation matrix for target pose
target_pose = np.eye(4, dtype=np.float32)
target_pose[:3, 3] = demo_sample["target_position"].cpu().numpy()
target_pose[:3, :3] = demo_sample["target_orientation"].cpu().numpy()

viz_client.publish_ghost_robot(robot.unnormalize_joints(demo_sample["configuration"]))
viz_client.publish_ghost_end_effector(pose=target_pose, frame=robot.tcp_link_name)

In [14]:
viz_client.clear_ghost_robot()
viz_client.clear_ghost_end_effector()

In [20]:
# viz_client demos: ghost meshes + point clouds
viz_client.publish_ghost_end_effector(
    pose=target_pose,
    frame=robot.tcp_link_name,
    color=[0.2, 0.8, 0.2],
    alpha=0.6,
 )
viz_client.publish_ghost_robot(
    robot.unnormalize_joints(demo_sample["configuration"]),
    color=[0.2, 0.6, 0.9],
    alpha=0.3,
 )

# Robot point cloud
viz_client.publish_robot_pointcloud(demo_sample["point_cloud"])

# Target and obstacle point clouds (centers only)
viz_client.publish_target_pointcloud(demo_sample["target_position"].unsqueeze(0))

obstacle_points = []
if demo_sample["cuboid_centers"].numel() > 0:
    obstacle_points.append(demo_sample["cuboid_centers"])
if demo_sample["cylinder_centers"].numel() > 0:
    obstacle_points.append(demo_sample["cylinder_centers"])
if obstacle_points:
    obstacle_points = torch.cat(obstacle_points, dim=0)
    viz_client.publish_obstacle_pointcloud(obstacle_points)

In [24]:
viz_client.clear_ghost_end_effector()
viz_client.clear_ghost_robot()
viz_client.clear_robot_pointcloud()
viz_client.clear_target_pointcloud()
viz_client.clear_obstacle_pointcloud()

# Visualize expert trajectories

In [16]:
viz_client.shutdown()
viz_client.connect(str(robot.urdf_path))

Sent shutdown command to viz_server
viz_server not running — starting…
Connected to viz_server


Visualize single expert trajectory,
2 animation modalities:
+ "interpolation": for smooth linear interpolation of the trajectory waypoints 
+ "discrete": for discrete steps between waypoints

In [25]:
animation_modality = "discrete"  # "interpolation" or "discrete"
sample_0 = val_dataset[2]

print(f"Animating expert trajectory...")
visualization.visualize_expert_trajectory(
    robot=robot,
    sample=sample_0,
    mode =animation_modality,
)

print("✓ Expert trajectory visualization started!")

Animating expert trajectory...
Original trajectory length: 60 states
Trimmed trajectory length: 41 states
✓ Expert trajectory visualization started!


Interactive slider to select different samples from the dataset and visualize their expert trajectories:

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create the widget objects
slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(val_dataset) - 1, # Dynamically set max based on your dataset size
    step=1,
    description='Sample Index:',
    continuous_update=False # Highly recommended for ROS/Robot vis
)

output = widgets.Output()

# Callback
def on_value_change(change):
    index = change['new']
    
    with output:
        clear_output(wait=True)
        print(f"Visualizing Sample: {index}")
        
        # Extract the specific sample from the dataset
        sample_i = val_dataset[index]  
        # Trigger ROS/Robot visualization
        visualization.visualize_expert_trajectory(robot=robot, sample=sample_i, mode=animation_modality)
        
        config_str = ", ".join([f"{angle:.3f}" for angle in sample_i['configuration'].numpy()])
        print(f"Robot starting configuration: [{config_str}]")
        print(f"✓ Visualization updated!")

# Link the slider to the function
slider.observe(on_value_change, names='value')
# Display
display(slider, output)
# Trigger the first sample immediately so the screen isn't blank
on_value_change({'new': slider.value})